# Question Answering

This notebook covers Question Answering (QA), a fundamental NLP task that involves answering questions based on a given context or document. QA systems are used in applications like virtual assistants, customer support bots, and information retrieval systems.

We will explore both extractive QA (finding answers in text) and generative QA (generating free-text answers).

In [ ]:
# Import required libraries
from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

## 1. Understanding Question Answering

Question Answering systems can be categorised in several ways:

**By Answer Type:**
- **Extractive**: Extracts a span of text from the context that answers the question
- **Abstractive**: Generates a free-text answer that may not appear verbatim in the context
- **Multiple Choice**: Selects the correct answer from given options

**By Knowledge Source:**
- **Closed-book**: Uses internal knowledge without external context
- **Open-book**: Uses provided context to find answers

Modern QA systems use transformer models like BERT, RoBERTa, and T5.

In [ ]:
# Load pre-trained QA model
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-uncased-distilled-squad"
)

print("=== QA Model Loaded ===")
print("Model: distilbert-base-uncased-distilled-squad")
print("This model is fine-tuned on the SQuAD dataset.")

## 2. Extractive Question Answering

Extractive QA identifies the exact span of text in the context that answers the question.

In [ ]:
# Sample context and questions
context = """
Natural language processing (NLP) is a subfield of linguistics, computer science, and artificial intelligence 
concerned with the interactions between computers and human language. It focuses on programming computers to 
process and analyse large amounts of natural language data. The result is a computer system that can understand 
patterns in language and derive meaning from human speech or written text.

NLP combines computational linguistics with machine learning and deep learning to create systems that can 
understand, interpret, and generate human language. Modern NLP applications include machine translation, 
sentiment analysis, chatbots, text summarisation, and named entity recognition.
"""

questions = [
    "What is NLP?",
    "What does NLP combine?",
    "What are modern NLP applications?",
    "What is the result of NLP?",
]

print("=== Context ===")
print(context)
print("\n=== Questions ===")
for q in questions:
    print(f"  - {q}")

In [ ]:
# Answer each question
print("=== Answers ===\n")

for question in questions:
    result = qa_pipeline(context=context, question=question)
    
    print(f"Question: {question}")
    print(f"Answer: {result['answer']}")
    print(f"Confidence: {result['score']:.4f}")
    print(f"Start: {result['start']}, End: {result['end']}")
    print()

## 3. Multiple Question Types

Let's test the QA system with different types of questions.

In [ ]:
# Different context for various question types
context2 = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. It is named after 
the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889 as the entrance 
arc for the 1889 World's Fair. The tower is 330 metres tall, about the same height as an 81-storey building.
Its height has increased by 24 metres due to the addition of various antennas. The tower weighs approximately 
10,100 tonnes. It is one of the most visited monuments in the world, with millions of visitors each year.
"""

question_types = {
    "What is the height?": "What is the height of the Eiffel Tower?",
    "Who is it named after?": "Who is the Eiffel Tower named after?",
    "When was it built?": "When was the Eiffel Tower built?",
    "How much does it weigh?": "How much does the Eiffel Tower weigh?",
    "Where is it located?": "Where is the Eiffel Tower located?",
}

print("=== Different Question Types ===\n")

for q_type, question in question_types.items():
    result = qa_pipeline(context=context2, question=question)
    print(f"{q_type}: {result['answer']}")

## 4. Generative Question Answering

For more natural answers, we can use generative models that produce free-text answers.

In [ ]:
# Load a text generation model for QA
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Use FLAN-T5 for generative QA
gen_model_name = "google/flan-t5-small"
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_model_name)

print("=== Generative QA Model ===")
print(f"Model: {gen_model_name}")

In [ ]:
def generative_qa(context, question, model, tokenizer, max_length=128):
    """Generate answer using a seq2seq model."""
    # Format input for FLAN-T5
    input_text = f"Context: {context} Question: {question} Answer:"
    
    # Tokenise
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    
    # Generate
    outputs = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )
    
    # Decode
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return answer

# Test generative QA
print("=== Generative QA Results ===\n")

for question in questions:
    answer = generative_qa(context, question, gen_model, gen_tokenizer)
    print(f"Question: {question}")
    print(f"Answer: {answer}\n")

## 5. Comparing Extractive vs Generative QA

Let's compare the two approaches on the same questions.

In [ ]:
# Compare approaches
print("=== Comparison: Extractive vs Generative QA ===\n")

test_questions = [
    "What is NLP?",
    "What does NLP combine?",
]

for q in test_questions:
    # Extractive
    extractive_result = qa_pipeline(context=context, question=q)
    
    # Generative
    generative_answer = generative_qa(context, q, gen_model, gen_tokenizer)
    
    print(f"Question: {q}")
    print(f"Extractive: {extractive_result['answer']} (score: {extractive_result['score']:.4f})")
    print(f"Generative: {generative_answer}")
    print()

## 6. Building a Simple QA System

Let's build a simple QA system that can answer questions from a knowledge base.

In [ ]:
# Simple knowledge base
knowledge_base = {
    "python": "Python is a high-level programming language created by Guido van Rossum and first released in 1991. It supports multiple programming paradigms and has a dynamic type system.",
    
    "machine learning": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing algorithms that can access data and learn from it.",
    
    "deep learning": "Deep learning is a subset of machine learning that uses artificial neural networks with multiple layers. It is inspired by the structure and function of the human brain and can process large amounts of unstructured data.",
    
    "transformer": "The transformer is a deep learning architecture introduced in 2017 based on the attention mechanism. It has become the foundation for many state-of-the-art NLP models like BERT and GPT.",
    
    "nlp": "Natural Language Processing (NLP) is a subfield of linguistics, computer science, and AI concerned with the interactions between computers and human language.",
}

def simple_qa_system(question, knowledge_base, qa_pipeline):
    """Simple QA system that finds relevant context."""
    question_lower = question.lower()
    
    # Find relevant context based on keywords
    best_context = None
    best_key = None
    
    for key, context in knowledge_base.items():
        if key in question_lower:
            best_context = context
            best_key = key
            break
    
    if best_context is None:
        # Try to find any relevant context
        for key, context in knowledge_base.items():
            if any(word in question_lower for word in key.split()):
                best_context = context
                best_key = key
                break
    
    if best_context:
        result = qa_pipeline(context=best_context, question=question)
        return result['answer'], best_key
    else:
        return "I don't have enough information to answer that question.", None

print("=== Simple QA System ===\n")

test_questions = [
    "What is Python?",
    "What is machine learning?",
    "What is a transformer?",
]

for q in test_questions:
    answer, context_key = simple_qa_system(q, knowledge_base, qa_pipeline)
    print(f"Question: {q}")
    print(f"Answer: {answer}")
    if context_key:
        print(f"Context from: {context_key}")
    print()

## Summary

In this notebook, we covered Question Answering:

1. **Understanding**: Learned about different types of QA systems
2. **Extractive QA**: Used DistilBERT for extractive question answering
3. **Question Types**: Tested various question types (who, what, when, where, how)
4. **Generative QA**: Used FLAN-T5 for generating free-text answers
5. **Comparison**: Compared extractive vs generative approaches
6. **QA System**: Built a simple QA system with a knowledge base

Question Answering is a crucial component of many AI applications. The choice between extractive and generative depends on your specific requirements for accuracy, speed, and answer format.